![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)







### Copyright notice

> <p><small><small>Copyright 2025 DeepMind Technologies Limited.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>

# Manipulation in The Playground! <a href="https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/manipulation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a>

In this notebook, we'll walk through a couple manipulation environments available in MuJoCo Playground.

**A Colab runtime with GPU acceleration is required.** If you're using a CPU-only runtime, you can switch using the menu "Runtime > Change runtime type".


In [1]:
#@title Install pre-requisites
#!pip install mujoco mujoco_mjx brax --quiet

### Check of CUDA and Apple Compatibility

In [2]:
#@title Install pre-requisites (CPU-only for Apple Silicon)
import os
import platform
import subprocess
import sys

def is_nvidia_available():
    """Check if nvidia-smi is callable (Linux/Colab with GPU)."""
    try:
        result = subprocess.run(
            ["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE
        )
        return result.returncode == 0
    except FileNotFoundError:
        return False

system = platform.system()
machine = platform.machine()

if system == "Darwin":  # macOS (Apple Silicon / Intel Mac)
    print("Detected macOS:", machine)
    # Force MuJoCo to use glfw rendering
    os.environ["MUJOCO_GL"] = "glfw"
    print("Using MUJOCO_GL=glfw for macOS")
    # Force JAX to CPU (avoid Metal/MPS backend issues)
    os.environ["JAX_PLATFORM_NAME"] = "cpu"
    print("Forcing JAX to run on CPU backend")

elif is_nvidia_available():  # Linux + NVIDIA GPU
    print("Detected NVIDIA GPU")
    os.environ["MUJOCO_GL"] = "egl"
    print("Using MUJOCO_GL=egl for GPU rendering")

    # Add missing EGL ICD config if necessary (Colab hack)
    NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
    if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
        with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
            f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")
else:  # Fallback (no GPU)
    print("No NVIDIA GPU detected, running in CPU/OSMesa mode")
    os.environ["MUJOCO_GL"] = "osmesa"
    os.environ["JAX_PLATFORM_NAME"] = "cpu"

# --- Test Mujoco ---
try:
    import mujoco
    mujoco.MjModel.from_xml_string('<mujoco/>')
    print("Mujoco installation and rendering backend OK")
except Exception as e:
    print("❌ Mujoco test failed:", e)

Detected macOS: arm64
Using MUJOCO_GL=glfw for macOS
Forcing JAX to run on CPU backend
Mujoco installation and rendering backend OK


In [3]:
# # @title Check if MuJoCo installation was successful

# import distutils.util
# import os
# import subprocess

# if subprocess.run('nvidia-smi').returncode:
#   raise RuntimeError(
#       'Cannot communicate with GPU. '
#       'Make sure you are using a GPU Colab runtime. '
#       'Go to the Runtime menu and select Choose runtime type.'
#   )

# # Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# # This is usually installed as part of an Nvidia driver package, but the Colab
# # kernel doesn't install its driver via APT, and as a result the ICD is missing.
# # (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
# NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
# if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
#   with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
#     f.write("""{
#     "file_format_version" : "1.0.0",
#     "ICD" : {
#         "library_path" : "libEGL_nvidia.so.0"
#     }
# }
# """)

# # Configure MuJoCo to use the EGL rendering backend (requires GPU)
# print('Setting environment variable to use GPU rendering:')
# %env MUJOCO_GL=egl

# try:
#   print('Checking that the installation succeeded:')
#   import mujoco

#   mujoco.MjModel.from_xml_string('<mujoco/>')
# except Exception as e:
#   raise e from RuntimeError(
#       'Something went wrong during installation. Check the shell output above '
#       'for more information.\n'
#       'If using a hosted Colab runtime, make sure you enable GPU acceleration '
#       'by going to the Runtime menu and selecting "Choose runtime type".'
#   )

# print('Installation successful.')

# # Tell XLA to use Triton GEMM, this improves steps/sec by ~30% on some GPUs
# xla_flags = os.environ.get('XLA_FLAGS', '')
# xla_flags += ' --xla_gpu_triton_gemm_any=True'
# os.environ['XLA_FLAGS'] = xla_flags

In [4]:
# # @title Import packages for plotting and creating graphics
# import json
# import itertools
# import time
# from typing import Callable, List, NamedTuple, Optional, Union
# import numpy as np

# # Graphics and plotting.
# print("Installing mediapy:")
# !command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
# !pip install -q mediapy
# import mediapy as media
# import matplotlib.pyplot as plt

# # More legible printing from numpy.
# np.set_printoptions(precision=3, suppress=True, linewidth=100)

### Check Critical Package Version
✅ JAX: 0.6.2
✅ MuJoCo: 3.3.6
✅ Brax: 0.13.0
✅ Flax: 0.10.7

In [5]:
# Version check
import jax, mujoco, brax, flax
print("JAX:", jax.__version__)
print("MuJoCo:", mujoco.__version__)
print("Brax:", brax.__version__)
print("Flax:", flax.__version__)


Failed to import warp: No module named 'warp'
Failed to import mujoco.mjx.third_party.mujoco_warp as mujoco_warp: No module named 'warp'
JAX: 0.6.2
MuJoCo: 3.3.6
Brax: 0.13.0
Flax: 0.10.7


In [6]:
# @title Import packages for plotting and creating graphics
import json
import itertools
import time
from typing import Callable, List, NamedTuple, Optional, Union
import numpy as np

# Graphics and plotting.
print("Installing mediapy & ffmpeg if needed...")


system = platform.system()
if system == "Linux":
    # Works in Colab/Ubuntu
    !command -v ffmpeg >/dev/null || (apt-get update -qq && apt-get install -y ffmpeg)
elif system == "Darwin":
    # On macOS, assume ffmpeg is installed via Homebrew, or ask user
    !command -v ffmpeg >/dev/null || echo " Please install ffmpeg with: brew install ffmpeg"

# !brew install ffmpeg
# !pip install -q mediapy
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

Installing mediapy & ffmpeg if needed...


In [7]:
# @title Import MuJoCo, MJX, and Brax
from datetime import datetime
import functools
import os
from typing import Any, Dict, Sequence, Tuple, Union
from brax import base
from brax import envs
from brax import math
from brax.base import Base, Motion, Transform
from brax.base import State as PipelineState
from brax.envs.base import Env, PipelineEnv, State
from brax.io import html, mjcf, model
from brax.mjx.base import State as MjxState
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo
from brax.training.agents.sac import networks as sac_networks
from brax.training.agents.sac import train as sac
from etils import epath
from flax import struct
from flax.training import orbax_utils
from IPython.display import HTML, clear_output
import jax
from jax import numpy as jp
from matplotlib import pyplot as plt
import mediapy as media
from ml_collections import config_dict
import mujoco
import mujoco.viewer
from mujoco import mjx
import numpy as np
from orbax import checkpoint as ocp
import os
import pickle
import numpy as np

## Number of CPU Cores used

to be able to utilze a multiple of 8 in the num_env, batch and minibatch size. we need to utilize only 8 and not 14 cores

In [8]:
import jax
import os

# must be done BEFORE "import jax"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"


devices = jax.devices()
device_count = len(devices)
print(devices)
print("JAX device count:", device_count)


[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7)]
JAX device count: 8


# Manipulation

MuJoCo Playground contains several manipulation environments (all listed below after running the command).

In [9]:
# #@title Install MuJoCo Playground
#!pip install playground


In [10]:
print("MuJoCo:", mujoco.__version__)
print("MJX ready:", hasattr(mjx.Data, "_impl"))

# MuJoCo: 3.1.4
# MJX ready: True

MuJoCo: 3.3.6
MJX ready: False


In [11]:
#@title Import The Playground

from mujoco_playground import wrapper
from mujoco_playground import registry

# Manipulation

MuJoCo Playground contains several manipulation environments (all listed below after running the command).

In [12]:
registry.manipulation.ALL_ENVS

('AlohaHandOver',
 'AlohaSinglePegInsertion',
 'PandaPickCube',
 'UR10PickCube',
 'PandaPickCubeOrientation',
 'PandaPickCubeCartesian',
 'PandaOpenCabinet',
 'PandaRobotiqPushCube',
 'LeapCubeReorient',
 'LeapCubeRotateZAxis')

# Franka Emika Panda

Let's start off with the simplest environment, simply picking up a cube with the Franka Emika Panda.

In [13]:
env_name = 'UR10PickCube'
env = registry.load(env_name)
env_cfg = registry.get_default_config(env_name)
print("Loaded:", env)
print("Action size:", env.action_size)
print("Gripper site ID:", env._gripper_site)

m = env.mj_model

print("\nBodies:", [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_BODY, i) for i in range(m.nbody)])
print("Joints:", [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_JOINT, i) for i in range(m.njnt)])
print("Actuators:", env.mj_model.nu, [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_ACTUATOR, i) for i in range(m.nu)])
print("Sites:", [mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_SITE, i) for i in range(m.nsite)])
print(env.action_size)
print([env.mj_model.joint(i).name for i in range(env.mj_model.njnt)])

✓ Using keyframe: 'task_home'
  Initial qpos size: 15
  Robot joints: [-1.5708 -1.5708  1.5708 -1.5708 -1.5708  0.      0.      0.    ]
Loaded: <mujoco_playground._src.manipulation.my_ur10.ur10pick.UR10PickCube object at 0x14320d9c0>
Action size: 7
Gripper site ID: 1

Bodies: ['world', 'base', 'shoulder_link', 'upper_arm_link', 'forearm_link', 'wrist_1_link', 'wrist_2_link', 'wrist_3_link', 'robotiq_hande_mount', 'hande_left_finger', 'hande_right_finger', 'box', 'mocap_target']
Joints: ['shoulder_pan_joint', 'shoulder_lift_joint', 'elbow_joint', 'wrist_1_joint', 'wrist_2_joint', 'wrist_3_joint', 'hande_left_finger_joint', 'hande_right_finger_joint', None]
Actuators: 7 ['shoulder_pan', 'shoulder_lift', 'elbow', 'wrist_1', 'wrist_2', 'wrist_3', 'hande_fingers_actuator']
Sites: ['attachment_site', 'tcp']
7
['shoulder_pan_joint', 'shoulder_lift_joint', 'elbow_joint', 'wrist_1_joint', 'wrist_2_joint', 'wrist_3_joint', 'hande_left_finger_joint', 'hande_right_finger_joint', '']


/Users/matthiasweiss/miniconda3/envs/mujoco/lib/python3.10/site-packages/mujoco/mjx/_src/mesh.py:141: UserWarning: Mesh "coupler" has a coplanar face with more than 20 vertices. This may lead to performance issues and inaccuracies in collision detection. Consider decimating the mesh.
  warnings.warn(
/Users/matthiasweiss/miniconda3/envs/mujoco/lib/python3.10/site-packages/mujoco/mjx/_src/mesh.py:141: UserWarning: Mesh "hande" has a coplanar face with more than 20 vertices. This may lead to performance issues and inaccuracies in collision detection. Consider decimating the mesh.
  warnings.warn(


In [14]:
print("Actuators:", env.mj_model.nu)
print([env.mj_model.actuator(i).name for i in range(env.mj_model.nu)])

Actuators: 7
['shoulder_pan', 'shoulder_lift', 'elbow', 'wrist_1', 'wrist_2', 'wrist_3', 'hande_fingers_actuator']


In [15]:
env_cfg

action_repeat: 1
action_scale: 0.04
ctrl_dt: 0.02
episode_length: 150
impl: jax
nconmax: 196608
njmax: 128
reward_config:
  scales:
    box_target: 8.0
    gripper_box: 4.0
    no_floor_collision: 0.25
    robot_target_qpos: 0.3
sim_dt: 0.005

## Rollout before training


In [16]:
# '../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_scene.xml'
# '../mujoco_playground/_src/manipulation/my_ur10/xmls/test_grvity.xml'
# '/../mujoco_playground/external_deps/mujoco_menagerie/franka_emika_panda/scene.xml'
# '../../mujoco_playground/_src/manipulation/my_ur10/universal_robots_ur10e/ur10e.xml'

In [17]:
model = mujoco.MjModel.from_xml_path('../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_scene.xml')
data = mujoco.MjData(model)

# List all keyframes
print(f"Available keyframes ({model.nkey}):")
for i in range(model.nkey):
    key = model.key(i)
    print(f"  {i}: {key.name}")

# Load gravity test keyframe
key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, 'gravity_test')
print(f"\ngravity_test key_id: {key_id}")

if key_id >= 0:
    mujoco.mj_resetDataKeyframe(model, data, key_id)
    print("✓ Loaded gravity_test keyframe")
else:
    print("✗ gravity_test not found!")

print("\nGravity:", model.opt.gravity)
print("Initial qpos:", data.qpos[:8])
print("Initial ctrl:", data.ctrl)
print("\nThe robot is ALREADY in gravity_test position")
print("Just press Space to start simulation - it should fall!")


Available keyframes (4):
  0: gravity_test
  1: home
  2: vertical_home
  3: tucked

gravity_test key_id: 0
✓ Loaded gravity_test keyframe

Gravity: [ 0.    0.   -9.81]
Initial qpos: [ 0.  -1.9  1.2 -1.6 -1.6  0.   0.   0. ]
Initial ctrl: [0. 0. 0. 0. 0. 0. 0.]

The robot is ALREADY in gravity_test position
Just press Space to start simulation - it should fall!


In [18]:
# Don't load any keyframe - just use defaults
print("Without loading any keyframe:")
print(f"Initial qpos: {data.qpos}")
print(f"Initial ctrl: {data.ctrl}")
print(f"Initial qfrc_actuator (actuator forces): {data.qfrc_actuator}")

print("\n" + "="*60)

# Now simulate for a bit
print("\nSimulating with ctrl=0...")
for i in range(100):
    mujoco.mj_step(model, data)
    if i % 25 == 0:
        print(f"Step {i}: qpos[0:3] = {data.qpos[0:3]}")

print("\nFinal qpos:", data.qpos)
print("\nDoes the robot move toward qpos=0 position?")

Without loading any keyframe:
Initial qpos: [ 0.  -1.9  1.2 -1.6 -1.6  0.   0.   0. ]
Initial ctrl: [0. 0. 0. 0. 0. 0. 0.]
Initial qfrc_actuator (actuator forces): [0. 0. 0. 0. 0. 0. 0. 0.]


Simulating with ctrl=0...
Step 0: qpos[0:3] = [ 1.03095520e-04 -1.89892299e+00  1.19869382e+00]
Step 25: qpos[0:3] = [ 0.00666291 -1.54570036  0.74521969]
Step 50: qpos[0:3] = [ 0.00371942 -0.78030791 -0.00433966]
Step 75: qpos[0:3] = [-0.00672551 -0.14050665 -0.08019658]

Final qpos: [-0.00577523  0.03110926 -0.03961626  0.05663166  0.10804136 -0.00018061
  0.00722859 -0.0044411 ]

Does the robot move toward qpos=0 position?


In [19]:
env = registry.load('UR10PickCube')

# Check what happens before reset
print("Environment loaded")
print(f"Action size: {env.action_size}")

# Reset environment
state = env.reset(jax.random.PRNGKey(0))

print("\nAfter reset:")
print(f"qpos: {state.data.qpos[:8]}")
print(f"ctrl: {state.data.ctrl}")
print("\nIs ctrl set to task_home values?")

✓ Using keyframe: 'task_home'
  Initial qpos size: 15
  Robot joints: [-1.5708 -1.5708  1.5708 -1.5708 -1.5708  0.      0.      0.    ]
Environment loaded
Action size: 7

After reset:
qpos: [-1.5708 -1.5708  1.5708 -1.5708 -1.5708  0.      0.      0.    ]
ctrl: [-1.5708 -1.5708  1.5708 -1.5708 -1.5708  0.      0.    ]

Is ctrl set to task_home values?


In [20]:
import time

# Load gravity test keyframe
key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, 'gravity_test')
mujoco.mj_resetDataKeyframe(model, data, key_id)

print("Gravity:", model.opt.gravity)
print("Initial qpos:", data.qpos[:8])
print("Initial ctrl:", data.ctrl)

# Simulate for 1 second (200 steps)
print("\nSimulating gravity...")
for i in range(200):
    mujoco.mj_step(model, data)
    if i % 50 == 0:
        print(f"Step {i}: base height = {data.qpos[0]:.4f}, joint2 = {data.qpos[1]:.4f}")

print("\nFinal qpos:", data.qpos[:8])
print("\nIf gravity works, joints should have changed from initial values")

Gravity: [ 0.    0.   -9.81]
Initial qpos: [ 0.  -1.9  1.2 -1.6 -1.6  0.   0.   0. ]
Initial ctrl: [0. 0. 0. 0. 0. 0. 0.]

Simulating gravity...
Step 0: base height = 0.0001, joint2 = -1.8989
Step 50: base height = 0.0037, joint2 = -0.7803
Step 100: base height = -0.0055, joint2 = 0.0291
Step 150: base height = -0.0042, joint2 = 0.0135

Final qpos: [-3.41738359e-03  1.25735876e-02  3.23793634e-04  2.07585973e-03
  3.86929993e-03 -5.46658558e-05 -1.69237790e-04  1.76860970e-04]

If gravity works, joints should have changed from initial values


In [21]:
model = mujoco.MjModel.from_xml_path('../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_scene.xml')

print("Actuator information:")
for i in range(model.nu):
    act = model.actuator(i)
    print(f"\nActuator {i}: {act.name}")
    print(f"  Bias type: {act.biastype}")  # 1 = affine (position control)
    print(f"  Gain type: {act.gaintype}")  # 0 = fixed gain
    print(f"  Gain params: {act.gainprm}")
    print(f"  Bias params: {act.biasprm}")
    print(f"  Control range: {act.ctrlrange}")
    print(f"  Force range: {act.forcerange}")


Actuator information:

Actuator 0: shoulder_pan
  Bias type: [1]
  Gain type: [0]
  Gain params: [5000.    0.    0.    0.    0.    0.    0.    0.    0.    0.]
  Bias params: [    0. -5000.  -500.     0.     0.     0.     0.     0.     0.     0.]
  Control range: [-6.2831  6.2831]
  Force range: [-330.  330.]

Actuator 1: shoulder_lift
  Bias type: [1]
  Gain type: [0]
  Gain params: [5000.    0.    0.    0.    0.    0.    0.    0.    0.    0.]
  Bias params: [    0. -5000.  -500.     0.     0.     0.     0.     0.     0.     0.]
  Control range: [-6.2831  6.2831]
  Force range: [-330.  330.]

Actuator 2: elbow
  Bias type: [1]
  Gain type: [0]
  Gain params: [5000.    0.    0.    0.    0.    0.    0.    0.    0.    0.]
  Bias params: [    0. -5000.  -500.     0.     0.     0.     0.     0.     0.     0.]
  Control range: [-3.1415  3.1415]
  Force range: [-150.  150.]

Actuator 3: wrist_1
  Bias type: [1]
  Gain type: [0]
  Gain params: [5000.    0.    0.    0.    0.    0.    0.    0.  

In [22]:
# Load task_home keyframe
key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, 'task_home')
if key_id >= 0:
    mujoco.mj_resetDataKeyframe(model, data, key_id)
    print("✓ Loaded task_home keyframe")
else:
    print("✗ task_home not found")

print("\n" + "="*60)
print("INTERACTIVE VIEWER CONTROLS")
print("="*60)
print("Standard controls:")
print("  Left-click + drag:  Rotate camera")
print("  Right-click + drag: Pan camera")
print("  Scroll:            Zoom")
print("  Space:             Pause/Resume")
print("  Backspace:         Reset simulation")
print("  ESC:               Exit")
print("\nAdvanced controls (in GUI):")
print("  Click 'Control' tab to see actuator sliders")
print("  Move sliders to control each joint")
print("  Set all to 0 to disable actuators")
print("="*60)

print(f"\nGravity: {model.opt.gravity}")
print(f"Initial qpos: {data.qpos[:8]}")
print(f"Initial ctrl: {data.ctrl}")

# Launch viewer
mujoco.viewer.launch(model, data)

✗ task_home not found

INTERACTIVE VIEWER CONTROLS
Standard controls:
  Left-click + drag:  Rotate camera
  Right-click + drag: Pan camera
  Scroll:            Zoom
  Space:             Pause/Resume
  Backspace:         Reset simulation
  ESC:               Exit

Advanced controls (in GUI):
  Click 'Control' tab to see actuator sliders
  Move sliders to control each joint
  Set all to 0 to disable actuators

Gravity: [ 0.    0.   -9.81]
Initial qpos: [-3.41738359e-03  1.25735876e-02  3.23793634e-04  2.07585973e-03
  3.86929993e-03 -5.46658558e-05 -1.69237790e-04  1.76860970e-04]
Initial ctrl: [0. 0. 0. 0. 0. 0. 0.]


In [23]:
model = mujoco.MjModel.from_xml_path('../../mujoco_playground/_src/manipulation/my_ur10/universal_robots_ur10e/ur10e_torque_test.xml')
data = mujoco.MjData(model)

# Load keyframe
key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, 'task_home')
mujoco.mj_resetDataKeyframe(model, data, key_id)

# Set ctrl to zero (now it's torque, not position!)
data.ctrl[:] = 0

print("Actuators are now TORQUE-based")
print("ctrl=0 means zero torque → pure gravity fall")

mujoco.viewer.launch(model, data)

Actuators are now TORQUE-based
ctrl=0 means zero torque → pure gravity fall


In [ ]:
model = mujoco.MjModel.from_xml_path('../../mujoco_playground/_src/manipulation/my_ur10/xmls/mjx_scene.xml')
data = mujoco.MjData(model)

# List all keyframes
print(f"Available keyframes ({model.nkey}):")
for i in range(model.nkey):
    key = model.key(i)
    print(f"  {i}: {key.name}")

# Load gravity test keyframe
key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, 'gravity_test')
print(f"\ngravity_test key_id: {key_id}")

if key_id >= 0:
    mujoco.mj_resetDataKeyframe(model, data, key_id)
    print("✓ Loaded gravity_test keyframe")
else:
    print("✗ gravity_test not found!")

print("\nGravity:", model.opt.gravity)
print("Initial qpos:", data.qpos[:8])
print("Initial ctrl:", data.ctrl)
print("\nThe robot is ALREADY in gravity_test position")
print("Just press Space to start simulation - it should fall!")


## Train Policy

Let's train the pick cube policy and visualize rollouts. The policy takes roughly 3 minutes to train on an RTX 4090.

In [ ]:
from mujoco_playground.config import manipulation_params
ppo_params = manipulation_params.brax_ppo_config(env_name)
ppo_params

In [ ]:
from copy import deepcopy

# Make a copy so you don't overwrite the original defaults
fast_ppo_params = deepcopy(ppo_params)

# --- Speedup adjustments ---
fast_ppo_params["num_timesteps"] = 10_000_000 
# fast_ppo_params["learning_rate"] = 0.002        
# fast_ppo_params["episode_length"] = 200        
# fast_ppo_params["unroll_length"] = 10         

# (Optional) reduce parallel envs to lower compute requirements
# fast_ppo_params["num_envs"] = 2048              # from 8192 or 4096 or 2048
# print("Adjusted num_envs:", fast_ppo_params["num_envs"])

# (Optional) tweak batch sizes accordingly !! Needs to be 
# fast_ppo_params["batch_size"] = 256             
# fast_ppo_params["num_minibatches"] = 16         

ppo_params = fast_ppo_params
ppo_params

### PPO

In [ ]:
x_data, y_data, y_dataerr = [], [], []
times = [datetime.now()]


def progress(num_steps, metrics):
  clear_output(wait=True)

  times.append(datetime.now())
  x_data.append(num_steps)
  y_data.append(metrics["eval/episode_reward"])
  y_dataerr.append(metrics["eval/episode_reward_std"])

  plt.xlim([0, ppo_params["num_timesteps"] * 1.25])
  plt.xlabel("# environment steps")
  plt.ylabel("reward per episode")
  plt.title(f"y={y_data[-1]:.3f}")
  plt.errorbar(x_data, y_data, yerr=y_dataerr, color="blue")

  display(plt.gcf())

ppo_training_params = dict(ppo_params)
network_factory = ppo_networks.make_ppo_networks
if "network_factory" in ppo_params:
  del ppo_training_params["network_factory"]
  network_factory = functools.partial(
      ppo_networks.make_ppo_networks,
      **ppo_params.network_factory
  )

train_fn = functools.partial(
    ppo.train, **dict(ppo_training_params),
    network_factory=network_factory,
    progress_fn=progress,
    seed=1
)

In [ ]:
make_inference_fn, params, metrics = train_fn(
    environment=env,
    wrap_env_fn=wrapper.wrap_for_brax_training,
)
if y_data:
    import numpy as np
    best_idx = int(np.argmax(y_data))

### Metrics of Training
Rewards over 1000 yield decent results
With default training time reward is 1347 after 20'152'320

In [ ]:
print(f"time to jit: {times[1] - times[0]}")
print(f"time to train: {times[-1] - times[1]}")
print(f"Highest Reward: {y_data[best_idx]:.3f} ± {y_dataerr[best_idx]:.3f} at step {x_data[best_idx]}")

## Visualize Rollouts

In [ ]:
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)
jit_inference_fn = jax.jit(make_inference_fn(params, deterministic=True))

In [ ]:
rng = jax.random.PRNGKey(42)
rollout = []
n_episodes = 1

for _ in range(n_episodes):
  state = jit_reset(rng)
  rollout.append(state)
  for i in range(env_cfg.episode_length):
    act_rng, rng = jax.random.split(rng)
    ctrl, _ = jit_inference_fn(state.obs, act_rng)
    state = jit_step(state, ctrl)
    rollout.append(state)

render_every = 1
frames = env.render(rollout[::render_every])
rewards = [s.reward for s in rollout]
media.show_video(frames, fps=1.0 / env.dt / render_every)

While the above policy is very simple, the work was extended using the Madrona batch renderer, and policies were transferred on a real robot. We encourage folks to check out the Madrona-MJX tutorial notebooks ([part 1](https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/training_vision_1.ipynb) and [part 2](https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/learning/notebooks/training_vision_2.ipynb))!